# 04 — Resultados y conclusiones

Comparación de señales, análisis cuantitativo y síntesis de los hallazgos del TPI.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import librosa
import librosa.display
from pathlib import Path

plt.rcParams['figure.dpi'] = 100
DATA_DIR = Path('../data')

## 1. Comparación de múltiples señales

In [ ]:
archivos = sorted(DATA_DIR.glob('*.wav'))
señales = []
nombres = []

for ruta in archivos:
    y, sr = librosa.load(ruta, sr=None)
    señales.append(y)
    nombres.append(ruta.stem)

print(f'Total de señales cargadas: {len(señales)}')

In [ ]:
# Panel de espectros comparados
n = min(len(señales), 4)
fig, axes = plt.subplots(n, 1, figsize=(14, 3*n), sharex=True)
if n == 1:
    axes = [axes]

for ax, y, nombre in zip(axes, señales[:n], nombres[:n]):
    Y = np.fft.rfft(y)
    freqs = np.fft.rfftfreq(len(y), d=1/sr)
    mag_db = 20 * np.log10(np.abs(Y) + 1e-10)
    ax.plot(freqs, mag_db, linewidth=0.6)
    ax.set_ylabel('dB')
    ax.set_title(nombre)
    ax.set_xlim(0, 8000)

axes[-1].set_xlabel('Frecuencia [Hz]')
plt.suptitle('Comparación de espectros de amplitud', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('../figuras/04_comparacion_espectros.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Tabla de métricas por señal

In [ ]:
print(f'{'Señal':<25} {'Dur[s]':>8} {'RMS':>8} {'F0[Hz]':>9} {'BW[Hz]':>9}')
print('-' * 65)

for y, nombre in zip(señales, nombres):
    dur = len(y) / sr
    rms = np.sqrt(np.mean(y**2))
    
    # F0 estimado
    Y = np.abs(np.fft.rfft(y))
    freqs = np.fft.rfftfreq(len(y), d=1/sr)
    mascara = (freqs >= 80) & (freqs <= 500)
    f0 = freqs[mascara][np.argmax(Y[mascara])]
    
    # Ancho de banda espectral (centroide de frecuencia)
    bw = librosa.feature.spectral_bandwidth(y=y, sr=sr).mean()
    
    print(f'{nombre:<25} {dur:>8.2f} {rms:>8.4f} {f0:>9.1f} {bw:>9.1f}')

## 3. Panel final para el informe

In [ ]:
# Panel 2x2 con los resultados principales de la primera señal
y, sr = librosa.load(archivos[0], sr=None)
n_fft, hop_length = 2048, 512

fig = plt.figure(figsize=(16, 10))
gs = gridspec.GridSpec(2, 2, figure=fig)

# Forma de onda
ax1 = fig.add_subplot(gs[0, 0])
t = np.linspace(0, len(y)/sr, len(y))
ax1.plot(t, y, linewidth=0.4, color='steelblue')
ax1.set_title('Forma de onda'); ax1.set_xlabel('Tiempo [s]'); ax1.set_ylabel('Amplitud')

# Espectro FFT
ax2 = fig.add_subplot(gs[0, 1])
Y = np.fft.rfft(y)
freqs = np.fft.rfftfreq(len(y), d=1/sr)
ax2.plot(freqs, 20*np.log10(np.abs(Y)+1e-10), linewidth=0.5, color='darkorange')
ax2.set_title('Espectro FFT'); ax2.set_xlabel('Frecuencia [Hz]'); ax2.set_ylabel('dB')
ax2.set_xlim(0, 8000)

# Espectrograma STFT
ax3 = fig.add_subplot(gs[1, 0])
D_db = librosa.amplitude_to_db(np.abs(librosa.stft(y, n_fft=n_fft, hop_length=hop_length)), ref=np.max)
librosa.display.specshow(D_db, sr=sr, hop_length=hop_length, x_axis='time', y_axis='hz', ax=ax3)
ax3.set_title('Espectrograma STFT'); ax3.set_ylim(0, 8000)

# Espectrograma Mel
ax4 = fig.add_subplot(gs[1, 1])
S_mel = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=n_fft, hop_length=hop_length, n_mels=128)
S_mel_db = librosa.power_to_db(S_mel, ref=np.max)
librosa.display.specshow(S_mel_db, sr=sr, hop_length=hop_length, x_axis='time', y_axis='mel', ax=ax4)
ax4.set_title('Espectrograma de Mel')

plt.suptitle(f'Resumen — {archivos[0].stem}', fontsize=15, y=1.01)
plt.tight_layout()
plt.savefig('../figuras/04_panel_final.png', dpi=150, bbox_inches='tight')
plt.show()
print('Panel guardado en figuras/04_panel_final.png')

## 4. Conclusiones

> **Completar con las conclusiones propias del análisis.**

- La FFT permitió identificar la frecuencia fundamental (F0) de las señales de voz analizadas, ubicada en el rango XX–YY Hz, consistent con voces [masculinas/femeninas].
- El espectrograma STFT reveló ...
- La escala de Mel permitió ...
- El filtro pasa-banda de 300–3400 Hz ...